In [8]:
import os
import torch
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Setup
# ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2  # ubah sesuai jumlah class (termasuk background)

model = fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.load_state_dict(torch.load("fasterrcnn_pascalvoc.pth", map_location=device))
model.to(device)
model.eval()

# ========================
# Inference ke gambar baru
# ========================
inference_dir = "dataset_pv/images"
image_files = [f for f in os.listdir(inference_dir) if f.endswith((".jpg", ".png"))]

for img_file in image_files:
    img_path = os.path.join(inference_dir, img_file)
    image = Image.open(img_path).convert("RGB")
    tensor = F.to_tensor(image).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(tensor)[0]

    img_np = tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    labels = prediction['labels'].cpu().numpy()



<ipython-input-9-36e31dfd9720>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("fasterrcnn_pascalvoc.pth", map_location=device))


In [ ]:
    # Visualisasi
    # ========================
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img_np)
    for box, score, label in zip(boxes, scores, labels):
        if score > 0.5:
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='cyan', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, ymin, f"Label {label}: {score:.2f}",
                    color='black', backgroundcolor='cyan', fontsize=8)
    plt.axis('off')
    plt.title(f"Inference: {img_file}")
    plt.show()